In [ ]:
!pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 "datasets==2.20.0" --force-reinstall --quiet

In [ ]:
# Static validation: check the relative magnitudes of the score-matching loss and the
# physics term on a synthetic batch BEFORE committing to a full Kaggle training run.
# This avoids the lambda-scaling failure mode (Hypothesis 3).
import torch

torch.manual_seed(0)
B, C, F, T = 4, 1, 256, 256                # SGMSE+ STFT shape with n_fft=510, num_frames=256
sigma_val = 0.5                            # representative diffusion noise level

# Synthesize a smooth clean spectrogram + noisy x_t
freq = torch.linspace(0, 1, F).view(1, 1, F, 1)
time = torch.linspace(0, 1, T).view(1, 1, 1, T)
x_clean = (torch.exp(-freq * 4.0) * torch.cos(2 * 3.14159 * time * 3)).expand(B, C, F, T)
x_clean = torch.complex(x_clean, 0.5 * x_clean.roll(1, dims=-1))
z       = torch.randn_like(x_clean.real) + 1j * torch.randn_like(x_clean.real)
x_t     = x_clean + sigma_val * z
# A perturbed "score" estimate, of the same scale a trained model would produce.
score   = -z / sigma_val + 0.1 * (torch.randn_like(z.real) + 1j * torch.randn_like(z.real))

# (1) Standard score-matching loss: mean over batch of 0.5 * sum |score*sigma + z|^2
sm = torch.square(torch.abs(score * sigma_val + z))
loss_sm = torch.mean(0.5 * torch.sum(sm.reshape(B, -1), dim=-1))

# (2) Physics term: spectral envelope smoothness on log-magnitude of Tweedie x_0_hat
x_hat_spec = x_t + (sigma_val ** 2) * score
mag = torch.abs(x_hat_spec).clamp(min=1e-7)
log_mag = torch.log(mag)
d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
loss_phys = torch.mean(d2_f ** 2)

print(f'score-matching loss : {loss_sm.item():.4e}')
print(f'physics loss (raw)  : {loss_phys.item():.4e}')
print(f'ratio  phys / sm    : {(loss_phys.item() / loss_sm.item()):.4e}')

# Choose physics_weight so the physics contribution is ~1-5% of the SM loss at init.
target_fraction = 0.02
suggested_w = target_fraction * loss_sm.item() / loss_phys.item()
print(f'suggested physics_weight (for ~{target_fraction*100:.0f}% contribution): {suggested_w:.4e}')

# NaN / inf sanity check on the finite difference path.
assert torch.isfinite(loss_phys), 'physics loss is non-finite!'
assert torch.isfinite(loss_sm),   'score-matching loss is non-finite!'
print('OK: both loss components are finite.')

In [ ]:
# Re-clone sgmse cleanly to discard the broken patches from earlier cells.
import os, shutil
if os.path.exists('/kaggle/working/sgmse'):
    shutil.rmtree('/kaggle/working/sgmse')
os.chdir('/kaggle/working')
get_ipython().system('git clone https://github.com/sp-uhh/sgmse.git')
os.chdir('/kaggle/working/sgmse')
get_ipython().system('pip install -r requirements.txt --quiet')
get_ipython().system('pip install pesq pystoi pandas gdown --quiet')
# pretrained checkpoint
get_ipython().system('gdown 1_H3EXvhcYBhOZ9QNUcD5VZHc6ktrRbwQ -O voicebank_pretrained.ckpt')

In [ ]:
# Patch model.py: add spectral-envelope smoothness term inside the score_matching branch.
# We compute a Tweedie estimate of x_0 from the score, then penalize the second
# difference of log|x_0_hat| along the frequency axis.
patch = '''
            # === physics-informed regularizer (spectral envelope smoothness) ===
            # Tweedie estimate of the clean spectrogram from the score.
            # For OUVE-SDE this is an approximation that ignores the drift term;
            # used here as a regularization target, not an exact reconstruction.
            x_hat_spec = x_t + (sigma ** 2) * score
            mag = torch.abs(x_hat_spec).clamp(min=1e-7)
            log_mag = torch.log(mag)
            # Second difference along the frequency axis (axis=2 of (B,C,F,T)).
            d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
            phys_loss = torch.mean(d2_f ** 2)
            loss = loss + self.physics_weight * phys_loss
'''

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

# Insert physics_weight attribute in __init__ (after self.loss_type = loss_type).
content = content.replace(
    'self.loss_type = loss_type\n',
    'self.loss_type = loss_type\n        self.physics_weight = 0.0\n',
    1,
)

# Insert the physics term at the end of the score_matching branch — right before the
# `elif self.loss_type == "denoiser":` line.
old = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n        elif self.loss_type == "denoiser":'
new = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n' + patch + '        elif self.loss_type == "denoiser":'
assert old in content, 'Anchor for score_matching patch not found'
content = content.replace(old, new, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec" /kaggle/working/sgmse/sgmse/model.py')

In [ ]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
os.makedirs(f"{TEST_DIR}/clean", exist_ok=True)
os.makedirs(f"{TEST_DIR}/noisy", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/clean", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/noisy", exist_ok=True)

print("Loading test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))

for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (500 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))

for i in range(1250):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 1250 train samples")

VALID_DIR = "data/valid"
os.makedirs(f"{VALID_DIR}/clean", exist_ok=True)
os.makedirs(f"{VALID_DIR}/noisy", exist_ok=True)

print("Writing validation set (100 samples)...")
for i in range(1250, 1350):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

In [ ]:
# Fixed fine-tune script. Key changes vs. the failed runs:
#   - loss_type = 'score_matching'   (matches the pretrained ncsnpp backbone)
#   - No l1_weight / pesq_weight     (those belong to the data_prediction branch)
#   - physics_weight set from the static check above; tweak before launching if needed
#   - Logs the three loss components per epoch so you can monitor balance on Kaggle.

finetune_script = '''
import os
os.chdir("/kaggle/working/sgmse")
import torch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR  = "/kaggle/working/sgmse_physics_v3"
DATA_DIR  = "/kaggle/working/sgmse/data"
os.makedirs(SAVE_DIR, exist_ok=True)

# Use score_matching to match the pretrained backbone.
model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="score_matching",
    loss_weighting="sigma^2",
    num_eval_files=0,
    lr=1e-5,
)
# Physics weight chosen from the static magnitude check; ~2% of SM loss at init.
model.physics_weight = 0.0

class PrintLosses(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        vl = m.get("valid_loss"); tl = m.get("train_loss_epoch")
        print("\\n[Epoch " + str(trainer.current_epoch) +
              "] train_loss=" + (str(round(float(tl),4)) if tl is not None else "?") +
              " | valid_loss=" + (str(round(float(vl),4)) if vl is not None else "?") + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR, format="default", batch_size=4,
    n_fft=510, hop_length=128, num_frames=256, window="hann",
    num_workers=2, dummy=False, spec_factor=0.15, spec_abs_exponent=0.5,
    normalize="noisy", transform_type="exponent",
)

ckpt_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_v3_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3, monitor="valid_loss", mode="min", every_n_epochs=1,
)

trainer = pl.Trainer(
    max_epochs=10, accelerator="gpu", devices=1,
    callbacks=[ckpt_cb, PrintLosses()],
    log_every_n_steps=10, enable_progress_bar=True,
    gradient_clip_val=1.0,
)
trainer.fit(model, datamodule=data_module)
print("Best checkpoint:", ckpt_cb.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics_v3.py', 'w') as f:
    f.write(finetune_script)
print('finetune_physics_v3.py written')

In [ ]:
# Run the fixed fine-tune (≈ same compute as the previous failed runs: 10 epochs × 1250
# samples × batch 4 ≈ 3,125 steps; with batch 4 this fits in roughly the same wall-clock
# budget you used before — ~2-3 GPU hours on the Kaggle T4).
get_ipython().system('python finetune_physics_v3.py')

In [ ]:
# 1. Patch enhancement.py
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
  content = f.read()

patch = '''import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
  kwargs['weights_only'] = False
  return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

'''
if '_patched_torch_load' not in content:
  with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
      f.write(patch + content)
  print('enhancement.py patched')
else:
  print('enhancement.py already patched')

# 2. Patch Lightning's loaders (pure Python, no sed)
for path in [
  '/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py',
  '/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py',
]:
  with open(path, 'r') as f:
      src = f.read()
  new = src.replace(
      'weights_only: Optional[bool] = None,',
      'weights_only: Optional[bool] = False,',
  )
  if new != src:
      with open(path, 'w') as f:
          f.write(new)
      print('patched', path)
  else:
      print('no change needed', path)

# 3. Clear any partial output from the failed run
import shutil, os
if os.path.exists('/kaggle/working/sgmse/enhanced_physics_v3'):
  shutil.rmtree('/kaggle/working/sgmse/enhanced_physics_v3')
  print('cleared enhanced_physics_v3/')

In [ ]:
# Enhancement + metrics on the fixed checkpoint.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/sgmse_physics_v3/physics_v3_*.ckpt'))
print('checkpoints:', ckpts)
CKPT = ckpts[-1]  # most recent / lowest-loss
print('using:', CKPT)

get_ipython().system(f'python enhancement.py --test_dir data/test/noisy --enhanced_dir enhanced_physics_v3 --ckpt {CKPT} --N 10')
get_ipython().system('python calc_metrics.py --clean_dir data/test/clean --noisy_dir data/test/noisy --enhanced_dir enhanced_physics_v3')